# Fontys ICT - Synthetic Student Journal Simulator (MVP)
This notebook implements a simple 7-day simulation loop for two distinct ICT student personas. The simulation follows the strict operational chain: **Event -> LLM Action Choice -> State Modification -> LLM Diary Generation**.


In [1]:
import random
import json
import requests

#-Selecting ollama model and endpoint
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "llama3.2:1b"  

def call_ollama_safe(prompt):
    """Helper function to send a prompt to Ollama with a safety timeout."""
    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False
    }
    try:
        response = requests.post(OLLAMA_URL, json=payload, timeout=15)
        text = response.json().get("response", "").strip()
        return text if text else "[Error: Empty response from model]"
    except requests.exceptions.Timeout:
        return "[Error: Ollama timeout - model is responding too slowly]"
    except Exception as e:
        return f"[Error connecting to Ollama: {e}]"

print(f"Configuration complete. Target model set to: {MODEL_NAME}")

Configuration complete. Target model set to: llama3.2:1b


## 1. Defining Personas and Event Impacts
We initialize two unique student personas with baseline stats (0-100), emotional states (0-100), and an OCEAN (Big Five) personality profile mapped on a 0-10 scale. 

We also scale the core *EVENT_EFFECTS* from the build guide to a 0-100 range and incorporate custom emotional impacts for *happy*, *angry*, and *scared*.

In [2]:
#-Persona definitions
personas = {
    "student_001": {
        "name": "Bram",
        "description": "An extraverted first-year ICT student who is easily distracted by social activities but remains highly ambitious.",
        "ocean": {"O": 7, "C": 4, "E": 9, "A": 8, "N": 5},
        "state": {
            "stress": 30, "energy": 80, "motivation": 70,
            "happy": 75, "angry": 10, "scared": 20
        },
        "diary_history": []
    },
    "student_002": {
        "name": "Sophie",
        "description": "An introverted, highly structured student who loves software engineering but frequently experiences performance anxiety.",
        "ocean": {"O": 8, "C": 9, "E": 3, "A": 6, "N": 7},
        "state": {
            "stress": 50, "energy": 60, "motivation": 85,
            "happy": 60, "angry": 5, "scared": 40
        },
        "diary_history": []
    }
}

#-Defining event impacts
EVENT_EFFECTS = {
    "lecture": {"stress": 5, "energy": -15, "motivation": 5, "happy": 5, "angry": 0, "scared": 2},
    "deadline": {"stress": 25, "energy": -20, "motivation": -5, "happy": -15, "angry": 10, "scared": 25},
    "exam": {"stress": 35, "energy": -30, "motivation": -10, "happy": -20, "angry": 5, "scared": 35},
    "social": {"stress": -20, "energy": -15, "motivation": 15, "happy": 30, "angry": -5, "scared": -10},
    "rest": {"stress": -15, "energy": 25, "motivation": 5, "happy": 10, "angry": -5, "scared": -5}
}

print("Personas and environment rules successfully initialized.")

Personas and environment rules successfully initialized.


## 2. The 7-Day Simulation Timeline
We establish a fixed chronological schedule containing a realistic mix of academic responsibilities, social events, and designated rest periods that both students will navigate day-by-day.

In [3]:
timeline = [
    {"day": 1, "event": "lecture"},
    {"day": 2, "event": "lecture"},
    {"day": 3, "event": "deadline"},
    {"day": 4, "event": "rest"},
    {"day": 5, "event": "lecture"},
    {"day": 6, "event": "social"},
    {"day": 7, "event": "exam"}
]

print(f"Timeline configured for a sequence of {len(timeline)} simulated days.")

Timeline configured for a sequence of 7 simulated days.


## 3. The Core Simulation Engine 
The simulation executes sequentially for each student. Every day, the system performs the following actions:
1. **Contextual Action Selection:** The LLM receives the student profile, current stats, and event context to choose a micro-action matching their personality.
2. **State Updates:** The mathematical engine applies the core changes, runs natural decay/recovery functions, and clamps values strictly between 0 and 100.
3. **Grounded Diary Generation:** The LLM processes the complete execution log to author a realistic diary fragment in casual English student language.

In [4]:
for student_id, student in personas.items():
    print(f"LAUNCHING SIMULATION FOR: {student['name'].upper()}")
    
    for day_info in timeline:
        day = day_info["day"]
        event = day_info["event"]
        
        print(f"[Day {day}] Processing event: '{event}'...")
        
        current_stats = student["state"]
        prev_state = current_stats.copy()
        recent_diaries = "\n".join(student["diary_history"][-2:]) # Context windows to maintain continuity
        
        #-Getting action from LLM 
        action_prompt = f"""
        You are {student['name']}. Description: {student['description']}.
        Your Big Five personality profile is: {student['ocean']}.
        
        It is Day {day}. Today's scheduled event is: {event}.
        Your current internal metrics are: {current_stats}.
        Your immediate memory of the past days:
        {recent_diaries}
        
        Determine one single, highly specific micro-action or personal choice you make today that directly aligns with this event and your persona traits.
        Example: 'I turn off my phone to eliminate distractions' or 'I sit at the very back of the lecture hall to take a nap'.
        Provide ONLY the short action in one sentence, nothing else.
        """
        
        print(f"   [...] Fetching persona choice from {MODEL_NAME}...")
        chosen_action = call_ollama_safe(action_prompt)
        print(f"   [✓] Choice captured: \"{chosen_action}\"")
        
        effects = EVENT_EFFECTS[event]
        for stat, delta in effects.items():
            if stat in current_stats:
                current_stats[stat] += delta
        
        #-Recovery from sleeping
        current_stats["stress"] -= 2
        current_stats["energy"] += 2
        
        #-Make sure stats remain within 0-100 bounds
        for stat in current_stats:
            current_stats[stat] = max(0, min(100, current_stats[stat]))
            
        stat_changes_text = ", ".join([f"{k}: {prev_state[k]} -> {current_stats[k]}" for k in current_stats])
        
        #-Generating diary
        diary_prompt = f"""
        Write a brief, informal diary entry in English for {student['name']}.
        Context: {student['description']}
        Current Timeline Day: {day}
        Scheduled Event: {event}
        Your real action taken today: '{chosen_action}'
        Actual mathematical changes in your state and emotions: {stat_changes_text}
        
        Strict Guidelines:
        - Write in the first-person ('I'), utilizing casual student slang. Minor typos or fragmented sentences are completely acceptable.
        - Do NOT invent external events or occurrences that are not explicitly documented in the execution logs.
        - Mirror your internal state (stress levels, happiness, energy) directly within the vocabulary and tone of your prose.
        - Keep the entry concise, containing a maximum of 4 sentences.
        """
        
        print(f"   [...] Generating grounded journal entry...")
        diary_entry = call_ollama_safe(diary_prompt)
        student["diary_history"].append(f"Day {day} ({event}): {diary_entry}")
        print(f"   [✓] Journal entry received and logged.")
        
        print(f"\nSIMULATION LOG - DAY {day} COMPLETE:")
        print(f"   State Trajectory: {stat_changes_text}")
        print(f"   Generated Text: \"{diary_entry}\"")
        print(f"-"*60 + "\n")

LAUNCHING SIMULATION FOR: BRAM
[Day 1] Processing event: 'lecture'...
   [...] Fetching persona choice from llama3.2:1b...
   [✓] Choice captured: "I decide to bring an extra set of notes and handouts for the course to reduce stress during the lecture."
   [...] Generating grounded journal entry...
   [✓] Journal entry received and logged.

SIMULATION LOG - DAY 1 COMPLETE:
   State Trajectory: stress: 30 -> 33, energy: 80 -> 67, motivation: 70 -> 75, happy: 75 -> 80, angry: 10 -> 10, scared: 20 -> 22
   Generated Text: "Ugh, day one of ICT class is already off to a stressful start. I'm planning on bringing an extra set of notes and handouts just in case my brain goes haywire during that lecture. Feeling pretty stressed already with all the material and trying to keep up. Still got a long night ahead of me with maths homework to get through, but at least I can relax now knowing I've got some backup plans for the big day tomorrow."
--------------------------------------------------------